# 03 — Graph Dataset Building

Converts the feature-engineered transaction table into a compressed graph dataset ready for TGN training on Kaggle.

**Input:** `feature_engineered_transactions.csv`  
**Output:** `nft_graph_dataset.npz`

---

### What this notebook does

1. Maps wallet addresses to integer node IDs
2. Selects the 9 final log-transformed edge features
3. Applies a **strictly chronological 70/15/15 split** (no shuffling) to preserve temporal ordering and prevent data leakage
4. Saves everything as a single `.npz` file

### Output file contents (`nft_graph_dataset.npz`)

| Key | Description |
|-----|-------------|
| `src` | Source node IDs (int64) |
| `dst` | Destination node IDs (int64) |
| `timestamps` | Unix timestamps (int64) |
| `edge_feat` | Edge feature matrix, shape (2,713,386 × 9) (float32) |
| `labels` | Wash trading labels: 1 = wash trade, 0 = normal (int64) |
| `train_mask` | Boolean mask for training split (70%) |
| `val_mask` | Boolean mask for validation split (15%) |
| `test_mask` | Boolean mask for test split (15%) |

## 1. Load feature-engineered transactions

In [ ]:
import pandas as pd
import numpy as np

print("Loading feature-engineered dataset...")

df = pd.read_csv("feature_engineered_transactions.csv")

# Ensure chronological order
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded: {df.shape[0]:,} transactions, {df.shape[1]} columns")

## 2. Build node mapping (wallet address → integer ID)

In [ ]:
all_wallets = pd.concat([
    df["from_address"],
    df["to_address"]
]).unique()

wallet_to_id = {
    wallet: idx
    for idx, wallet in enumerate(all_wallets)
}

df["src"] = df["from_address"].map(wallet_to_id)
df["dst"] = df["to_address"].map(wallet_to_id)

print(f"Unique wallets (nodes): {len(wallet_to_id):,}")

## 3. Select edge features and build arrays

In [ ]:
# 9 final log-transformed edge features (matches paper methodology)
edge_features = [
    "log_transaction_value",
    "log_src_tx_count_past",
    "log_dst_tx_count_past",
    "log_src_to_dst_count_past",
    "log_dst_to_src_count_past",
    "log_time_since_src_last",
    "log_time_since_dst_last",
    "log_src_value_sum_past",
    "log_dst_value_sum_past"
]

print(f"Edge features ({len(edge_features)}): {edge_features}")

# Build numpy arrays
src        = df["src"].astype(np.int64).values
dst        = df["dst"].astype(np.int64).values
timestamps = df["timestamp"].astype(np.int64).values
labels     = df["is_wash_trading"].astype(np.int64).values
edge_feat  = df[edge_features].astype(np.float32).values

print(f"edge_feat shape: {edge_feat.shape}")

## 4. Chronological 70/15/15 split

Transactions are already sorted by timestamp. The split is applied by position index — no shuffling — to ensure the model is always tested on transactions that occur **after** the training period.

In [ ]:
n = len(df)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_mask = np.arange(n) < train_end
val_mask   = (np.arange(n) >= train_end) & (np.arange(n) < val_end)
test_mask  = np.arange(n) >= val_end

print("Dataset split summary:")
print(f"  Train      : {train_mask.sum():,} edges  |  Fraud: {labels[train_mask].sum():,}")
print(f"  Validation : {val_mask.sum():,} edges  |  Fraud: {labels[val_mask].sum():,}")
print(f"  Test       : {test_mask.sum():,} edges  |  Fraud: {labels[test_mask].sum():,}")
print(f"\n  Total nodes : {len(np.unique(np.concatenate([src, dst]))):,}")
print(f"  Total edges : {n:,}")
print(f"  Fraud ratio : {labels.mean():.4%}")

## 5. Save graph dataset

In [ ]:
np.savez_compressed(
    "nft_graph_dataset.npz",
    src=src,
    dst=dst,
    timestamps=timestamps,
    edge_feat=edge_feat,
    labels=labels,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

print("Saved: nft_graph_dataset.npz")
print("\nNext step: upload nft_graph_dataset.npz to Kaggle and run 04_tgn_training.ipynb")